# Covered-face detector: train YOLO on public Roboflow datasets

Run the cells **top to bottom**. Before starting: **Runtime → Change runtime type → T4 GPU**.

Classes the model learns:
- `0 face`: face visible (no mask)
- `1 covered_face`: mask or other covering (this triggers the alert)

At the end you download **`best.pt`** and upload it to your Hugging Face Space.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install packages

In [ ]:
!pip install -q ultralytics roboflow pyyaml

## 3. Save results to Google Drive (so a Colab disconnect doesn't lose your model)
Click **Connect** / **Allow** in the popup.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/mask_yolo'

## 4. Choose the datasets

1. Make a free account at https://universe.roboflow.com
2. Get your **API key**: avatar (top right) → **Settings** → **API Keys** → copy the *Private API Key*.
3. For each dataset: open it → **Dataset** → **Download Dataset** → format **YOLOv8** → **Show download code**.
   The code shows `rf.workspace("XXX").project("YYY")` and `.version(N)`: put those 3 values in the list below.

Four datasets are already filled in:
- two **mask** datasets
- **covered-face-detection** (about 17k images, covered / not-covered faces)
- **face-hiding-detection** (hand, scarf, balaclava, mask)

If one fails to download (renamed, removed, or a different version number), open it on Universe and copy the right values, or delete that line.

In [ ]:
ROBOFLOW_API_KEY = "PASTE_YOUR_ROBOFLOW_API_KEY_HERE"

DATASETS = [
    # (workspace, project, version)
    ("agh-ett2f", "mask-detection-yolov8", 1),                 # masks
    ("uk-zychs", "face-mask-detection-oryic", 1),              # masks
    ("covered-face-detection", "covered-face-detection", 2),   # covered / not-covered faces (big)
    ("university-8zqnr", "face-hiding-detection", 12),         # hand, scarf, balaclava, mask
]

## 5. Download the datasets

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

downloaded = []
for ws, proj, ver in DATASETS:
    try:
        ds = rf.workspace(ws).project(proj).version(ver).download("yolov8", location=f"/content/rf/{proj}")
        downloaded.append(ds.location)
        print("OK:", proj)
    except Exception as e:
        print("FAILED:", ws, proj, ver, "->", e)

assert downloaded, "No dataset downloaded - check API key and dataset names"

## 6. Convert all datasets to our 2 classes and merge them
Check the printed mapping. If a class is mapped wrong, add it to `MANUAL_MAP` and run this cell again.

In [ ]:
import shutil, yaml
from pathlib import Path

# Force a mapping here if the automatic one is wrong:  0 = face, 1 = covered_face, None = drop
MANUAL_MAP = {
    "hand": 1,                     # hand over the face
    "balaclava": 1,
    "scarf": 1,
    "concealing glasses": None,    # sunglasses cover less than 50% -> ignore
    "non-concealing glasses": None,
    "nothing": None,
    "with_helmet": None,           # a helmet alone doesn't hide the face
    "without_helmet": None,
}

def auto_map(name):
    n = name.lower().replace("-", "_").replace(" ", "_")
    if any(k in n for k in ["no_mask", "nomask", "without", "unmask", "no_face_mask", "bare",
                            "not_covered", "notcovered", "uncovered", "not_occlu", "visible"]):
        return 0
    if any(k in n for k in ["mask", "covered", "incorrect", "scarf", "occlu", "balaclava", "hidden"]):
        return 1
    if n in ["face", "person_face", "head"]:
        return 0
    return None

DST = Path("/content/dataset")
shutil.rmtree(DST, ignore_errors=True)
counts = {"train": 0, "val": 0}
boxes = {0: 0, 1: 0}

for src in map(Path, downloaded):
    names = yaml.safe_load((src / "data.yaml").read_text())["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    mapping = {i: MANUAL_MAP.get(n, auto_map(n)) for i, n in enumerate(names)}
    print(f"\n{src.name}:")
    for i, n in enumerate(names):
        print(f"   {n!r:28} -> {['face', 'covered_face'][mapping[i]] if mapping[i] is not None else 'DROPPED'}")

    for rf_split, our in {"train": "train", "valid": "val", "test": "val"}.items():
        img_dir, lbl_dir = src / rf_split / "images", src / rf_split / "labels"
        if not img_dir.exists():
            continue
        (DST / "images" / our).mkdir(parents=True, exist_ok=True)
        (DST / "labels" / our).mkdir(parents=True, exist_ok=True)
        for img in img_dir.iterdir():
            out = []
            lbl = lbl_dir / (img.stem + ".txt")
            if lbl.exists():
                for line in lbl.read_text().splitlines():
                    p = line.split()
                    if len(p) < 5 or mapping.get(int(p[0])) is None:
                        continue
                    new = mapping[int(p[0])]
                    if len(p) > 5:  # polygon -> box
                        xs, ys = [float(v) for v in p[1::2]], [float(v) for v in p[2::2]]
                        p = [p[0], (min(xs)+max(xs))/2, (min(ys)+max(ys))/2, max(xs)-min(xs), max(ys)-min(ys)]
                    out.append(f"{new} " + " ".join(f"{float(v):.6f}" for v in p[1:5]))
                    boxes[new] += 1
            name = f"{src.name}_{img.name}"          # prefix avoids name clashes between datasets
            shutil.copy(img, DST / "images" / our / name)
            (DST / "labels" / our / (Path(name).stem + ".txt")).write_text("\n".join(out))
            counts[our] += 1

print(f"\nTOTAL: {counts['train']} train / {counts['val']} val images")
print(f"Boxes: face={boxes[0]}  covered_face={boxes[1]}")

Path("/content/data.yaml").write_text(
    "path: /content/dataset\ntrain: images/train\nval: images/val\nnames:\n  0: face\n  1: covered_face\n")

## 7. Train
- `yolo11n` = nano (fast). The cloud server can also handle `yolo11s.pt` (more accurate, about 2x slower), so change `BASE_MODEL` if you want that.
- With all 4 datasets (about 20k images) this takes about **2–4 hours**. Keep this tab open.
  Everything is saved to Google Drive after each epoch.
- If Colab disconnects: run cells 2 and 3 again, then the **resume** cell below.

In [ ]:
from ultralytics import YOLO

BASE_MODEL = "yolo11n.pt"
IMG_SIZE = 640

model = YOLO(BASE_MODEL)
model.train(
    data="/content/data.yaml",
    epochs=60,            # big dataset -> 60 epochs is enough
    patience=20,          # stops early when it stops improving
    imgsz=IMG_SIZE,
    batch=32,
    project=SAVE_DIR,
    name="run",
    exist_ok=True,
    fliplr=0.5, hsv_v=0.4, degrees=10,
)

### Resume (only if training was interrupted)

In [ ]:
# from ultralytics import YOLO
# YOLO(f"{SAVE_DIR}/run/weights/last.pt").train(resume=True)

## 8. Check accuracy
`mAP50` above about 0.8 is good for this task. Look at `confusion_matrix.png` too.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display

best = YOLO(f"{SAVE_DIR}/run/weights/best.pt")
m = best.val(data="/content/data.yaml", imgsz=640)
print("mAP50:", round(m.box.map50, 3), " mAP50-95:", round(m.box.map, 3))
display(Image(f"{SAVE_DIR}/run/confusion_matrix.png", width=600))

## 9. Quick test on a few validation images

In [ ]:
import glob, random
imgs = random.sample(glob.glob("/content/dataset/images/val/*"), 4)
for r in best.predict(imgs, imgsz=640, conf=0.5, verbose=False):
    r.save(filename="/content/pred.jpg")
    display(Image("/content/pred.jpg", width=400))

## 10. Download the model
This downloads **`best.pt`** (about 5–6 MB). It is also saved in Google Drive under `MyDrive/mask_yolo/run/weights/`.
Upload it to your Hugging Face Space next (see README).

In [ ]:
from google.colab import files
files.download(f"{SAVE_DIR}/run/weights/best.pt")